# EPC Estimator for Caxton Villa

This notebook estimates the Energy Performance Certificate (EPC) rating for domestic flats in the UK.

EPC ratings range from A (most efficient) to G (least efficient), based on the property's energy efficiency and environmental impact.

## Key Factors for Flats

- **Building fabric**: Walls, roof, floors, windows
- **Heating system**: Type, efficiency, controls
- **Hot water**: System type and efficiency
- **Lighting**: Low-energy lighting percentage
- **Ventilation**: Natural vs mechanical
- **Property position**: Ground floor, mid-floor, top floor

In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import Optional, Tuple
from enum import Enum

# Set up plotting style
plt.style.use('default')
%matplotlib inline

# this is used in some of the classes below.  I don't know how to make it local to that class.
temp_diff = 20


## 1. Define Data Classes and Enums

Enhanced with detailed wall and roof construction types.

In [7]:
class FlatPosition(Enum):
    """Position of flat within building"""
    GROUND_FLOOR = "ground_floor"
    MID_FLOOR = "mid_floor"
    TOP_FLOOR = "top_floor"
    SINGLE_STOREY = "single_storey"


class WallType(Enum):
    """
    Wall construction type with various insulation levels.
    U-values based on typical UK building constructions (W/m²K).
    """
    # Solid walls (pre-1919 typical)
    SOLID_UNINSULATED = "solid_uninsulated"
    SOLID_50MM_INSULATION = "solid_50mm_insulation"
    SOLID_100MM_INSULATION = "solid_100mm_insulation"
    SOLID_150MM_INSULATION = "solid_150mm_insulation"
    
    # Cavity walls (1919-1990 typical)
    CAVITY_UNINSULATED = "cavity_uninsulated"
    CAVITY_PARTIAL_FILL = "cavity_partial_fill"
    CAVITY_FULL_FILL = "cavity_full_fill"
    
    # Modern constructions
    TIMBER_FRAME_UNINSULATED = "timber_frame_uninsulated"
    TIMBER_FRAME_INSULATED = "timber_frame_insulated"
    STEEL_FRAME_UNINSULATED = "steel_frame_uninsulated"
    STEEL_FRAME_INSULATED = "steel_frame_insulated"
    
    # System-built (post-war)
    SYSTEM_BUILT_UNINSULATED = "system_built_uninsulated"
    SYSTEM_BUILT_INSULATED = "system_built_insulated"


class RoofType(Enum):
    """
    Roof construction type for top-floor flats.
    Includes both pitched and flat roof options.
    U-values in W/m²K.
    """
    # Pitched roofs (loft space above)
    PITCHED_NO_INSULATION = "pitched_no_insulation"
    PITCHED_50MM_LOFT_INSULATION = "pitched_50mm_loft_insulation"
    PITCHED_100MM_LOFT_INSULATION = "pitched_100mm_loft_insulation"
    PITCHED_200MM_LOFT_INSULATION = "pitched_200mm_loft_insulation"
    PITCHED_270MM_LOFT_INSULATION = "pitched_270mm_loft_insulation"
    PITCHED_ROOM_IN_ROOF_UNINSULATED = "pitched_room_in_roof_uninsulated"
    PITCHED_ROOM_IN_ROOF_INSULATED = "pitched_room_in_roof_insulated"
    
    # Flat roofs
    FLAT_UNINSULATED = "flat_uninsulated"
    FLAT_INSULATED = "flat_insulated"
    FLAT_WARM_DECK = "flat_warm_deck"
    FLAT_INVERTED = "flat_inverted"
    
    # No roof (not top floor)
    NONE = "none"


class FloorType(Enum):
    """
    Floor construction type for ground-floor flats.
    U-values in W/m²K.
    """
    SOLID_NO_INSULATION = "solid_no_insulation"
    SOLID_INSULATED = "solid_insulated"
    SUSPENDED_NO_INSULATION = "suspended_no_insulation"
    SUSPENDED_INSULATED = "suspended_insulated"
    ABOVE_UNHEATED_SPACE = "above_unheated_space"
    NONE = "none"  # Not ground floor


class HeatingType(Enum):
    """Primary heating system"""
    GAS_BOILER_OLD = "gas_boiler_old"
    GAS_BOILER_MODERATE = "gas_boiler_moderate"
    GAS_BOILER_EFFICIENT = "gas_boiler_efficient"
    ELECTRIC_STORAGE = "electric_storage"
    ELECTRIC_DIRECT = "electric_direct"
    HEAT_PUMP = "heat_pump"
    DISTRICT_HEATING = "district_heating"
    COMMUNITY_HEATING = "community_heating"


class WindowType(Enum):
    """Window glazing type"""
    SINGLE_GLAZED = "single_glazed"
    DOUBLE_GLAZED_OLD = "double_glazed_old"
    DOUBLE_GLAZED_NEW = "double_glazed_new"
    TRIPLE_GLAZED = "triple_glazed"
    SECONDARY_GLAZING = "secondary_glazing"


class HotWaterType(Enum):
    """Hot water system"""
    FROM_HEATING = "from_heating"
    ELECTRIC_IMMERSION = "electric_immersion"
    GAS_INSTANT = "gas_instant"
    SOLAR_THERMAL = "solar_thermal"


@dataclass
class FlatProperties:
    """Physical properties of the flat"""
    floor_area_sqm: float = 40.0
    position: FlatPosition = FlatPosition.GROUND_FLOOR
    num_bedrooms: int = 1
    num_occupants: int = 2
    year_built: int = 2026
    
    # Wall properties
    wall_type: WallType = WallType.CAVITY_FULL_FILL
    wall_area_sqm: Optional[float] = None  # Auto-calculated if None
    num_external_walls: int = 2  # Typical for flats
    
    # Roof properties (for top floor flats)
    roof_type: RoofType = RoofType.NONE
    roof_area_sqm: Optional[float] = None  # Auto-calculated if None
    
    # Floor properties (for ground floor flats)
    floor_type: FloorType = FloorType.SOLID_INSULATED
    floor_area_exposed_sqm: Optional[float] = None  # Auto-calculated if None
    
    # Windows
    window_type: WindowType = WindowType.DOUBLE_GLAZED_NEW
    window_area_sqm: Optional[float] = None  # Auto-calculated if None
    
    # Heating and hot water
    heating_type: HeatingType = HeatingType.GAS_BOILER_EFFICIENT
    hot_water_type: HotWaterType = HotWaterType.GAS_INSTANT
    
    # Controls
    has_cylinder_thermostat: bool = True
    has_room_thermostat: bool = True
    has_trvs: bool = True
    has_programmer: bool = True
    
    # Lighting and renewables
    low_energy_lighting_pct: float = 100.0
    has_solar_pv: bool = False
    solar_pv_kw: float = 0.0
    
    # Party wall/flat separation
    flat_below_insulated: bool = True
    flat_above_insulated: bool = True
    party_walls_insulated: bool = False
    
    # Thermal bridging
    thermal_bridge_factor: float = 0.15  # Default 15% additional heat loss
    
    # Air tightness
    air_changes_per_hour: float = 0.5  # Default for older buildings
    has_draught_proofing: bool = True


## 2. EPC Calculation Engine with Detailed Wall and Roof Handling

In [8]:
class EPCEstimator:
    """
    EPC Estimator with detailed wall and roof calculations.
    Based on SAP methodology principles.
    """
    
    # U-values for different wall constructions (W/m²K)
    # Based on typical UK building stock and SAP 2012/10.2
    WALL_U_VALUES = {
        # Solid walls
        WallType.SOLID_UNINSULATED: 2.10,
        WallType.SOLID_50MM_INSULATION: 0.70,
        WallType.SOLID_100MM_INSULATION: 0.45,
        WallType.SOLID_150MM_INSULATION: 0.30,
        
        # Cavity walls
        WallType.CAVITY_UNINSULATED: 1.50,
        WallType.CAVITY_PARTIAL_FILL: 0.55,
        WallType.CAVITY_FULL_FILL: 0.35,
        
        # Timber frame
        WallType.TIMBER_FRAME_UNINSULATED: 1.20,
        WallType.TIMBER_FRAME_INSULATED: 0.35,
        
        # Steel frame
        WallType.STEEL_FRAME_UNINSULATED: 1.40,
        WallType.STEEL_FRAME_INSULATED: 0.40,
        
        # System built
        WallType.SYSTEM_BUILT_UNINSULATED: 2.00,
        WallType.SYSTEM_BUILT_INSULATED: 0.45,
    }
    
    # U-values for different roof constructions (W/m²K)
    ROOF_U_VALUES = {
        # Pitched roofs with loft insulation
        RoofType.PITCHED_NO_INSULATION: 2.30,
        RoofType.PITCHED_50MM_LOFT_INSULATION: 0.70,
        RoofType.PITCHED_100MM_LOFT_INSULATION: 0.40,
        RoofType.PITCHED_200MM_LOFT_INSULATION: 0.25,
        RoofType.PITCHED_270MM_LOFT_INSULATION: 0.16,
        RoofType.PITCHED_ROOM_IN_ROOF_UNINSULATED: 2.30,
        RoofType.PITCHED_ROOM_IN_ROOF_INSULATED: 0.30,
        
        # Flat roofs
        RoofType.FLAT_UNINSULATED: 2.20,
        RoofType.FLAT_INSULATED: 0.35,
        RoofType.FLAT_WARM_DECK: 0.28,
        RoofType.FLAT_INVERTED: 0.25,
        
        # No roof
        RoofType.NONE: 0.0,
    }
    
    # U-values for different floor constructions (W/m²K)
    FLOOR_U_VALUES = {
        FloorType.SOLID_NO_INSULATION: 0.70,
        FloorType.SOLID_INSULATED: 0.25,
        FloorType.SUSPENDED_NO_INSULATION: 0.90,
        FloorType.SUSPENDED_INSULATED: 0.30,
        FloorType.ABOVE_UNHEATED_SPACE: 0.50,
        FloorType.NONE: 0.0,
    }
    
    # Window U-values (W/m²K)
    WINDOW_U_VALUES = {
        WindowType.SINGLE_GLAZED: 4.80,
        WindowType.DOUBLE_GLAZED_OLD: 3.10,
        WindowType.DOUBLE_GLAZED_NEW: 1.80,
        WindowType.TRIPLE_GLAZED: 1.00,
        WindowType.SECONDARY_GLAZING: 2.80,
    }
    
    # Heating efficiencies
    HEATING_EFFICIENCIES = {
        HeatingType.GAS_BOILER_OLD: 0.65,
        HeatingType.GAS_BOILER_MODERATE: 0.78,
        HeatingType.GAS_BOILER_EFFICIENT: 0.90,
        HeatingType.ELECTRIC_STORAGE: 1.0,
        HeatingType.ELECTRIC_DIRECT: 1.0,
        HeatingType.HEAT_PUMP: 3.0,
        HeatingType.DISTRICT_HEATING: 0.85,
        HeatingType.COMMUNITY_HEATING: 0.80,
    }
    
    # Fuel costs (pence per kWh)
    FUEL_COSTS = {
        HeatingType.GAS_BOILER_OLD: 7.0,
        HeatingType.GAS_BOILER_MODERATE: 7.0,
        HeatingType.GAS_BOILER_EFFICIENT: 7.0,
        HeatingType.ELECTRIC_STORAGE: 30.0,
        HeatingType.ELECTRIC_DIRECT: 30.0,
        HeatingType.HEAT_PUMP: 10.0,
        HeatingType.DISTRICT_HEATING: 8.0,
        HeatingType.COMMUNITY_HEATING: 8.0,
    }
    
    # CO2 emission factors (kg CO2 per kWh)
    CO2_FACTORS = {
        HeatingType.GAS_BOILER_OLD: 0.210,
        HeatingType.GAS_BOILER_MODERATE: 0.210,
        HeatingType.GAS_BOILER_EFFICIENT: 0.210,
        HeatingType.ELECTRIC_STORAGE: 0.136,
        HeatingType.ELECTRIC_DIRECT: 0.136,
        HeatingType.HEAT_PUMP: 0.045,
        HeatingType.DISTRICT_HEATING: 0.150,
        HeatingType.COMMUNITY_HEATING: 0.180,
    }

    # temp diff for heat loss calcs

    def __init__(self, flat: FlatProperties):
        self.flat = flat
    
    def calculate_wall_area(self) -> float:
        """
        Calculate external wall area based on floor area and number of external walls.
        Assumes a typical room height of 2.5m.
        """
        if self.flat.wall_area_sqm is not None:
            return self.flat.wall_area_sqm
        
        area = self.flat.floor_area_sqm
        height = 2.5
        
        # Estimate room dimensions from floor area
        # Assume rectangular shape
        if self.flat.num_external_walls == 1:
            # Flat in middle of terrace/block, one external wall
            width = np.sqrt(area * 1.5)
            external_wall_length = width
        elif self.flat.num_external_walls == 2:
            # Corner flat or typical flat with front and side external
            width = np.sqrt(area)
            depth = area / width
            external_wall_length = width + depth
        elif self.flat.num_external_walls == 3:
            # End of terrace or top floor with roof exposure
            width = np.sqrt(area * 0.8)
            depth = area / width
            external_wall_length = width + 2 * depth
        else:
            # Detached or all walls external
            width = np.sqrt(area)
            depth = area / width
            external_wall_length = 2 * (width + depth)
        
        return external_wall_length * height
    
    def calculate_roof_area(self) -> float:
        """
        Calculate roof area for top-floor flats.
        Returns 0 for non-top-floor flats unless explicitly set.
        """
        if self.flat.roof_area_sqm is not None:
            return self.flat.roof_area_sqm
        
        # Only top floor flats have roof exposure
        if self.flat.position != FlatPosition.TOP_FLOOR:
            return 0.0
        
        return self.flat.floor_area_sqm
    
    def calculate_exposed_floor_area(self) -> float:
        """
        Calculate exposed floor area for ground-floor flats.
        Returns 0 for flats above heated spaces.
        """
        if self.flat.floor_area_exposed_sqm is not None:
            return self.flat.floor_area_exposed_sqm
        
        # Ground floor flats have exposed floor
        if self.flat.position == FlatPosition.GROUND_FLOOR:
            return self.flat.floor_area_sqm
        
        # Single storey has exposed floor
        if self.flat.position == FlatPosition.SINGLE_STOREY:
            return self.flat.floor_area_sqm
        
        return 0.0
    
    def calculate_window_area(self) -> float:
        """
        Calculate window area as percentage of floor area.
        Typical range: 15-25% of floor area.
        """
        if self.flat.window_area_sqm is not None:
            return self.flat.window_area_sqm
        
        # Default: 18% of floor area
        return self.flat.floor_area_sqm * 0.18
    
    def get_effective_floor_type(self) -> FloorType:
        """
        Determine the effective floor type based on flat position.
        """
        if self.flat.position in [FlatPosition.MID_FLOOR, FlatPosition.TOP_FLOOR]:
            # Mid and top floors typically have flat below
            if self.flat.flat_below_insulated:
                return FloorType.NONE
            else:
                return FloorType.ABOVE_UNHEATED_SPACE
        return self.flat.floor_type
    
    def get_effective_roof_type(self) -> RoofType:
        """
        Determine the effective roof type based on flat position.
        """
        if self.flat.position == FlatPosition.TOP_FLOOR:
            return self.flat.roof_type
        elif self.flat.position == FlatPosition.SINGLE_STOREY:
            return self.flat.roof_type
        return RoofType.NONE
    
    def calculate_fabric_heat_loss(self) -> dict:
        """
        Calculate heat loss through building fabric (walls, roof, floor, windows).
        Returns detailed breakdown of heat losses.
        """
        temp_diff = 20  # Temperature difference (internal - external)
        
        # Get areas
        wall_area = self.calculate_wall_area()
        window_area = self.calculate_window_area()
        roof_area = self.calculate_roof_area()
        floor_area = self.calculate_exposed_floor_area()
        
        # Adjust wall area to exclude windows
        net_wall_area = max(0, wall_area - window_area)
        
        # Get U-values
        wall_u = self.WALL_U_VALUES[self.flat.wall_type]
        window_u = self.WINDOW_U_VALUES[self.flat.window_type]
        
        effective_roof_type = self.get_effective_roof_type()
        roof_u = self.ROOF_U_VALUES[effective_roof_type]
        
        effective_floor_type = self.get_effective_floor_type()
        floor_u = self.FLOOR_U_VALUES[effective_floor_type]
        
        # Calculate heat losses (W/K)
        wall_loss_wk = net_wall_area * wall_u
        window_loss_wk = window_area * window_u
        roof_loss_wk = roof_area * roof_u
        floor_loss_wk = floor_area * floor_u
        
        # Apply thermal bridging factor
        thermal_bridge_multiplier = 1 + self.flat.thermal_bridge_factor
        
        # Calculate heat loss at temperature difference
        wall_loss = wall_loss_wk * temp_diff * thermal_bridge_multiplier
        window_loss = window_loss_wk * temp_diff
        roof_loss = roof_loss_wk * temp_diff
        floor_loss = floor_loss_wk * temp_diff
        
        # Party wall heat loss (if not insulated)
        party_wall_loss = 0
        if not self.flat.party_walls_insulated:
            # Estimate party wall area (simplified)
            party_wall_area = self.flat.floor_area_sqm * 0.5  # Rough estimate
            party_wall_u = 0.5  # Typical for uninsulated party wall
            party_wall_loss = party_wall_area * party_wall_u * temp_diff * 0.1  # Small factor
        
        total_fabric_loss = wall_loss + window_loss + roof_loss + floor_loss + party_wall_loss
        
        return {
            'wall_area_sqm': net_wall_area,
            'wall_u_value': wall_u,
            'wall_loss_wk': wall_loss_wk,
            'wall_loss_w': wall_loss,
            'window_area_sqm': window_area,
            'window_u_value': window_u,
            'window_loss_wk': window_loss_wk,
            'window_loss_w': window_loss,
            'roof_area_sqm': roof_area,
            'roof_u_value': roof_u,
            'roof_loss_wk': roof_loss_wk,
            'roof_loss_w': roof_loss,
            'floor_area_sqm': floor_area,
            'floor_u_value': floor_u,
            'floor_loss_wk': floor_loss_wk,
            'floor_loss_w': floor_loss,
            'party_wall_loss_w': party_wall_loss,
            'thermal_bridge_factor': self.flat.thermal_bridge_factor,
            'total_fabric_loss_w': total_fabric_loss,
        }
    
    def calculate_ventilation_heat_loss(self) -> dict:
        """
        Calculate heat loss due to ventilation/air infiltration.
        """
        
        air_density = 1.2  # kg/m³
        specific_heat = 1005  # J/kg·K
        
        # Volume of the flat
        height = 2.5
        volume = self.flat.floor_area_sqm * height
        
        # Air changes per hour
        air_changes = self.flat.air_changes_per_hour
        
        # Apply draught proofing reduction
        if self.flat.has_draught_proofing:
            air_changes *= 0.7  # 30% reduction
        
        # Ventilation heat loss (W)
        vent_loss_w = (volume * air_changes * air_density * specific_heat * temp_diff) / 3600
        
        return {
            'volume_m3': volume,
            'air_changes_per_hour': air_changes,
            'ventilation_loss_w': vent_loss_w,
        }
    
    def calculate_heat_loss(self) -> dict:
        """
        Calculate total heat loss for the flat.
        Combines fabric and ventilation heat losses.
        """
        
        fabric = self.calculate_fabric_heat_loss()
        ventilation = self.calculate_ventilation_heat_loss()
        
        total_loss = fabric['total_fabric_loss_w'] + ventilation['ventilation_loss_w']
        
        return {
            **fabric,
            **ventilation,
            'total_loss_w': total_loss,
            'heat_loss_parameter_wk': total_loss / temp_diff,
        }
    
    def calculate_annual_energy(self) -> dict:
        """
        Calculate annual energy requirements.
        """
        heat_loss = self.calculate_heat_loss()
        
        # Heating degree days approximation
        # UK typical: ~2000 degree days per year
        heating_hours = 200 * 24 * 0.6  # Approximate heating season hours
        
        # Space heating requirement
        space_heating_kwh = (heat_loss['total_loss_w'] * heating_hours) / 1000
        
        # Apply heating controls factor
        control_factor = 1.0
        if self.flat.has_room_thermostat:
            control_factor -= 0.10
        if self.flat.has_trvs:
            control_factor -= 0.05
        if self.flat.has_cylinder_thermostat:
            control_factor -= 0.05
        if self.flat.has_programmer:
            control_factor -= 0.05
        
        space_heating_kwh *= max(0.7, control_factor)
        
        # Hot water energy
        hot_water_kwh = self.flat.floor_area_sqm * 50
        
        # Lighting energy
        lighting_kwh = self.flat.floor_area_sqm * 10
        lighting_kwh *= (1 - (self.flat.low_energy_lighting_pct / 100) * 0.75)
        
        # Appliances and cooking
        appliance_kwh = 500 + (self.flat.num_occupants * 300)
        cooking_kwh = 200 + (self.flat.num_occupants * 100)
        
        # Solar PV generation
        solar_generation = 0
        if self.flat.has_solar_pv:
            solar_generation = self.flat.solar_pv_kw * 850  # Typical UK yield
        
        total_consumption = space_heating_kwh + hot_water_kwh + lighting_kwh + appliance_kwh + cooking_kwh
        net_consumption = total_consumption - solar_generation
        
        return {
            'space_heating_kwh': space_heating_kwh,
            'hot_water_kwh': hot_water_kwh,
            'lighting_kwh': lighting_kwh,
            'appliance_kwh': appliance_kwh,
            'cooking_kwh': cooking_kwh,
            'solar_generation_kwh': solar_generation,
            'total_consumption_kwh': total_consumption,
            'net_consumption_kwh': net_consumption,
        }
    
    def calculate_epc(self) -> dict:
        """
        Calculate EPC rating and related metrics.
        """
        energy = self.calculate_annual_energy()
        heat_loss = self.calculate_heat_loss()
        
        heating_eff = self.HEATING_EFFICIENCIES[self.flat.heating_type]
        fuel_cost_rate = self.FUEL_COSTS[self.flat.heating_type]
        co2_factor = self.CO2_FACTORS[self.flat.heating_type]
        
        # Delivered energy (accounting for system efficiency)
        space_heating_delivered = energy['space_heating_kwh'] / heating_eff
        
        if self.flat.hot_water_type == HotWaterType.FROM_HEATING:
            hot_water_delivered = energy['hot_water_kwh'] / heating_eff
        elif self.flat.hot_water_type in [HotWaterType.ELECTRIC_IMMERSION]:
            hot_water_delivered = energy['hot_water_kwh']
        else:
            hot_water_delivered = energy['hot_water_kwh'] / 0.8
        
        total_delivered = (space_heating_delivered + hot_water_delivered + 
                          energy['lighting_kwh'] + energy['appliance_kwh'] + energy['cooking_kwh'])
        primary_energy_per_sqm = total_delivered / self.flat.floor_area_sqm
        
        # CO2 calculations
        space_heating_co2 = space_heating_delivered * co2_factor
        hot_water_co2 = hot_water_delivered * co2_factor
        other_co2 = (energy['lighting_kwh'] + energy['appliance_kwh'] + energy['cooking_kwh']) * 0.136
        total_co2 = space_heating_co2 + hot_water_co2 + other_co2
        co2_per_sqm = total_co2 / self.flat.floor_area_sqm
        
        # Cost calculations
        heating_cost = (space_heating_delivered + hot_water_delivered) * fuel_cost_rate / 100
        other_cost = (energy['lighting_kwh'] + energy['appliance_kwh'] + energy['cooking_kwh']) * 30 / 100
        total_cost = heating_cost + other_cost
        
        # EPC rating bands
        if co2_per_sqm <= 5:
            rating_band = 'A'
            rating_score = 92 + min(8, (5 - co2_per_sqm) * 2)
        elif co2_per_sqm <= 15:
            rating_band = 'B'
            rating_score = 81 + min(11, (15 - co2_per_sqm) * 1.1)
        elif co2_per_sqm <= 30:
            rating_band = 'C'
            rating_score = 69 + min(12, (30 - co2_per_sqm) * 0.8)
        elif co2_per_sqm <= 50:
            rating_band = 'D'
            rating_score = 55 + min(14, (50 - co2_per_sqm) * 0.7)
        elif co2_per_sqm <= 70:
            rating_band = 'E'
            rating_score = 39 + min(16, (70 - co2_per_sqm) * 0.8)
        elif co2_per_sqm <= 90:
            rating_band = 'F'
            rating_score = 21 + min(18, (90 - co2_per_sqm) * 0.9)
        else:
            rating_band = 'G'
            rating_score = max(1, 21 - (co2_per_sqm - 90) * 0.5)
        
        rating_score = round(rating_score)
        
        return {
            'rating_band': rating_band,
            'rating_score': rating_score,
            'primary_energy_kwh_m2': round(primary_energy_per_sqm, 1),
            'co2_emissions_kg_m2': round(co2_per_sqm, 1),
            'total_co2_kg_year': round(total_co2, 0),
            'estimated_annual_cost_gbp': round(total_cost, 0),
            'energy_breakdown': energy,
            'heat_loss_breakdown': heat_loss,
            'space_heating_delivered_kwh': round(space_heating_delivered, 0),
            'hot_water_delivered_kwh': round(hot_water_delivered, 0),
        }
    
    def get_fabric_summary(self) -> pd.DataFrame:
        """
        Get a summary of fabric elements as a DataFrame.
        """
        heat_loss = self.calculate_heat_loss()
        
        data = {
            'Element': ['Walls', 'Windows', 'Roof', 'Floor'],
            'Area (m²)': [
                round(heat_loss['wall_area_sqm'], 1),
                round(heat_loss['window_area_sqm'], 1),
                round(heat_loss['roof_area_sqm'], 1),
                round(heat_loss['floor_area_sqm'], 1),
            ],
            'U-value (W/m²K)': [
                heat_loss['wall_u_value'],
                heat_loss['window_u_value'],
                heat_loss['roof_u_value'],
                heat_loss['floor_u_value'],
            ],
            'Heat Loss (W/K)': [
                round(heat_loss['wall_loss_wk'], 1),
                round(heat_loss['window_loss_wk'], 1),
                round(heat_loss['roof_loss_wk'], 1),
                round(heat_loss['floor_loss_wk'], 1),
            ],
        }
        
        return pd.DataFrame(data)

## 3. Caxton Villa Calculations with Detailed Wall and Roof Analysis

In [9]:
# Define Caxton Villa flat configurations with detailed wall and roof specifications

def create_cv_flat(flat_number: int) -> FlatProperties:
    """
    Create FlatProperties for a specific Caxton Villa flat.
    
    Flats 1-6 configuration:
    - Flats 1, 4: Solid uninsulated walls (older construction)
    - Flats 2, 3, 5, 6: Cavity insulated walls
    - Flat 6: Top floor with roof exposure
    - Flats 4, 5: Mid floor
    - Flats 1, 2, 3: Ground floor
    """
    
    # Wall type assignment
    if flat_number in [1, 4]:
        wall_type = WallType.SOLID_UNINSULATED
    else:
        wall_type = WallType.CAVITY_FULL_FILL
    
    # Position assignment
    if flat_number == 6:
        position = FlatPosition.TOP_FLOOR
        roof_type = RoofType.PITCHED_100MM_LOFT_INSULATION
    elif flat_number in [4, 5]:
        position = FlatPosition.MID_FLOOR
        roof_type = RoofType.NONE
    else:
        position = FlatPosition.GROUND_FLOOR
        roof_type = RoofType.NONE
    
    # Floor type based on position
    if position == FlatPosition.GROUND_FLOOR:
        floor_type = FloorType.SOLID_INSULATED
    else:
        floor_type = FloorType.NONE
    
    return FlatProperties(
        floor_area_sqm=40,
        position=position,
        wall_type=wall_type,
        roof_type=roof_type,
        floor_type=floor_type,
        num_external_walls=2,
        window_type=WindowType.DOUBLE_GLAZED_NEW,
        heating_type=HeatingType.GAS_BOILER_EFFICIENT,
        hot_water_type=HotWaterType.GAS_INSTANT,
        has_room_thermostat=True,
        has_trvs=True,
        thermal_bridge_factor=0.15,
        air_changes_per_hour=0.5,
    )


# Create all flats
cv_flats = [create_cv_flat(i) for i in range(1, 7)]

In [10]:
# Calculate and display results for all Caxton Villa flats
results = []

for i, flat in enumerate(cv_flats, 1):
    estimator = EPCEstimator(flat)
    result = estimator.calculate_epc()
    results.append(result)
    
    print(f"=== Caxton Villa Flat {i} ({flat.position.value.replace('_', ' ').title()}) ===")
    print(f"Wall Type: {flat.wall_type.name} (U-value: {estimator.WALL_U_VALUES[flat.wall_type]} W/m²K)")
    
    roof_type_display = flat.roof_type.name if flat.roof_type != RoofType.NONE else 'NONE'
    print(f"Roof Type: {roof_type_display}")
    
    floor_desc = flat.floor_type.name
    if flat.floor_type == FloorType.NONE:
        floor_desc += ' (above heated space)'
    print(f"Floor Type: {floor_desc}")
    
    print("\nFabric Summary:")
    fabric_df = estimator.get_fabric_summary()
    print(fabric_df.to_string(index=False))
    
    print(f"\nEPC Rating: {result['rating_band']} ({result['rating_score']}/100)")
    print(f"Primary Energy: {result['primary_energy_kwh_m2']} kWh/m²/year")
    print(f"CO2 Emissions: {result['co2_emissions_kg_m2']} kg/m²/year")
    print(f"Est. Annual Cost: £{result['estimated_annual_cost_gbp']}")
    print()

=== Caxton Villa Flat 1 (Ground Floor) ===
Wall Type: SOLID_UNINSULATED (U-value: 2.1 W/m²K)
Roof Type: NONE
Floor Type: SOLID_INSULATED

Fabric Summary:
Element  Area (m²)  U-value (W/m²K)  Heat Loss (W/K)
  Walls       24.4             2.10             51.3
Windows        7.2             1.80             13.0
   Roof        0.0             0.00              0.0
  Floor       40.0             0.25             10.0

EPC Rating: D (60/100)
Primary Energy: 216.1 kWh/m²/year
CO2 Emissions: 42.4 kg/m²/year
Est. Annual Cost: £973.0

=== Caxton Villa Flat 2 (Ground Floor) ===
Wall Type: CAVITY_FULL_FILL (U-value: 0.35 W/m²K)
Roof Type: NONE
Floor Type: SOLID_INSULATED

Fabric Summary:
Element  Area (m²)  U-value (W/m²K)  Heat Loss (W/K)
  Walls       24.4             0.35              8.5
Windows        7.2             1.80             13.0
   Roof        0.0             0.00              0.0
  Floor       40.0             0.25             10.0

EPC Rating: D (69/100)
Primary Energy: 157.1 k

## 4. Wall Insulation Impact Analysis

Compare different wall insulation scenarios for the same flat.

In [11]:
def compare_wall_types(base_flat: FlatProperties, wall_types: list) -> pd.DataFrame:
    """
    Compare EPC ratings for different wall types.
    """
    comparisons = []
    
    for wall_type in wall_types:
        test_flat = FlatProperties(
            floor_area_sqm=base_flat.floor_area_sqm,
            position=base_flat.position,
            num_bedrooms=base_flat.num_bedrooms,
            num_occupants=base_flat.num_occupants,
            year_built=base_flat.year_built,
            wall_type=wall_type,
            roof_type=base_flat.roof_type,
            floor_type=base_flat.floor_type,
            window_type=base_flat.window_type,
            heating_type=base_flat.heating_type,
            hot_water_type=base_flat.hot_water_type,
            has_room_thermostat=base_flat.has_room_thermostat,
            has_trvs=base_flat.has_trvs,
            low_energy_lighting_pct=base_flat.low_energy_lighting_pct,
        )
        
        estimator = EPCEstimator(test_flat)
        result = estimator.calculate_epc()
        heat_loss = estimator.calculate_heat_loss()
        
        comparisons.append({
            'Wall Type': wall_type.name.replace('_', ' ').title(),
            'U-value (W/m²K)': estimator.WALL_U_VALUES[wall_type],
            'Wall Heat Loss (W/K)': round(heat_loss['wall_loss_wk'], 1),
            'Total Heat Loss (W/K)': round(heat_loss['heat_loss_parameter_wk'], 1),
            'EPC Band': result['rating_band'],
            'EPC Score': result['rating_score'],
            'Annual Cost (£)': result['estimated_annual_cost_gbp'],
        })
    
    return pd.DataFrame(comparisons)


# Base flat for comparison
base_flat = FlatProperties(
    floor_area_sqm=40,
    position=FlatPosition.GROUND_FLOOR,
    num_bedrooms=1,
    num_occupants=2,
    floor_type=FloorType.SOLID_INSULATED,
    window_type=WindowType.DOUBLE_GLAZED_NEW,
    heating_type=HeatingType.GAS_BOILER_EFFICIENT,
    hot_water_type=HotWaterType.GAS_INSTANT,
    has_room_thermostat=True,
    has_trvs=True,
)

# Solid wall options
solid_wall_types = [
    WallType.SOLID_UNINSULATED,
    WallType.SOLID_50MM_INSULATION,
    WallType.SOLID_100MM_INSULATION,
    WallType.SOLID_150MM_INSULATION,
]

# Cavity wall options
cavity_wall_types = [
    WallType.CAVITY_UNINSULATED,
    WallType.CAVITY_PARTIAL_FILL,
    WallType.CAVITY_FULL_FILL,
]

In [12]:
# Compare solid wall options
print("Solid Wall Insulation Comparison:")
print()
solid_comparison = compare_wall_types(base_flat, solid_wall_types)
print(solid_comparison.to_string())
print()

# Compare cavity wall options
print("Cavity Wall Insulation Comparison:")
print()
cavity_comparison = compare_wall_types(base_flat, cavity_wall_types)
print(cavity_comparison.to_string())
print()

Solid Wall Insulation Comparison:

                Wall Type  U-value (W/m²K)  Wall Heat Loss (W/K)  Total Heat Loss (W/K) EPC Band  EPC Score  Annual Cost (£)
0       Solid Uninsulated             2.10                  51.3                   94.7        D         60            973.0
1   Solid 50Mm Insulation             0.70                  17.1                   55.3        D         67            841.0
2  Solid 100Mm Insulation             0.45                  11.0                   48.3        D         68            817.0
3  Solid 150Mm Insulation             0.30                   7.3                   44.1        C         69            803.0

Cavity Wall Insulation Comparison:

             Wall Type  U-value (W/m²K)  Wall Heat Loss (W/K)  Total Heat Loss (W/K) EPC Band  EPC Score  Annual Cost (£)
0   Cavity Uninsulated             1.50                  36.6                   77.8        D         63            916.0
1  Cavity Partial Fill             0.55                  13

## 5. Roof Insulation Impact Analysis

Compare different roof insulation scenarios for top-floor flats.

In [ ]:
def compare_roof_types(base_flat: FlatProperties, roof_types: list) -> pd.DataFrame:
    """
    Compare EPC ratings for different roof types.
    """
    comparisons = []
    
    for roof_type in roof_types:
        test_flat = FlatProperties(
            floor_area_sqm=base_flat.floor_area_sqm,
            position=base_flat.position,
            num_bedrooms=base_flat.num_bedrooms,
            num_occupants=base_flat.num_occupants,
            year_built=base_flat.year_built,
            wall_type=base_flat.wall_type,
            roof_type=roof_type,
            floor_type=base_flat.floor_type,
            window_type=base_flat.window_type,
            heating_type=base_flat.heating_type,
            hot_water_type=base_flat.hot_water_type,
            has_room_thermostat=base_flat.has_room_thermostat,
            has_trvs=base_flat.has_trvs,
            low_energy_lighting_pct=base_flat.low_energy_lighting_pct,
        )
        
        estimator = EPCEstimator(test_flat)
        result = estimator.calculate_epc()
        heat_loss = estimator.calculate_heat_loss()
        
        comparisons.append({
            'Roof Type': roof_type.name.replace('_', ' ').title(),
            'U-value (W/m²K)': estimator.ROOF_U_VALUES[roof_type],
            'Roof Heat Loss (W/K)': round(heat_loss['roof_loss_wk'], 1),
            'Total Heat Loss (W/K)': round(heat_loss['heat_loss_parameter_wk'], 1),
            'EPC Band': result['rating_band'],
            'EPC Score': result['rating_score'],
            'Annual Cost (£)': result['estimated_annual_cost_gbp'],
        })
    
    return pd.DataFrame(comparisons)


# Top floor flat for roof comparison
top_floor_flat = FlatProperties(
    floor_area_sqm=40,
    position=FlatPosition.TOP_FLOOR,
    num_bedrooms=1,
    num_occupants=2,
    wall_type=WallType.CAVITY_FULL_FILL,
    roof_type=RoofType.PITCHED_100MM_LOFT_INSULATION,
    floor_type=FloorType.NONE,
    window_type=WindowType.DOUBLE_GLAZED_NEW,
    heating_type=HeatingType.GAS_BOILER_EFFICIENT,
    hot_water_type=HotWaterType.GAS_INSTANT,
    has_room_thermostat=True,
    has_trvs=True,
)

# Pitched roof options
pitched_roof_types = [
    RoofType.PITCHED_NO_INSULATION,
    RoofType.PITCHED_50MM_LOFT_INSULATION,
    RoofType.PITCHED_100MM_LOFT_INSULATION,
    RoofType.PITCHED_200MM_LOFT_INSULATION,
    RoofType.PITCHED_270MM_LOFT_INSULATION,
]

# Flat roof options
flat_roof_types = [
    RoofType.FLAT_UNINSULATED,
    RoofType.FLAT_INSULATED,
    RoofType.FLAT_WARM_DECK,
    RoofType.FLAT_INVERTED,
]

In [ ]:
# Compare pitched roof options
print("Pitched Roof Insulation Comparison:")
print()
pitched_comparison = compare_roof_types(top_floor_flat, pitched_roof_types)
print(pitched_comparison.to_string())
print()

# Compare flat roof options
print("Flat Roof Insulation Comparison:")
print()
flat_roof_comparison = compare_roof_types(top_floor_flat, flat_roof_types)
print(flat_roof_comparison.to_string())
print()

## 6. Combined Wall and Roof Improvement Analysis

Analyze the combined impact of improving both walls and roof.

In [ ]:
def analyze_improvements(base_flat: FlatProperties, improvements: list) -> pd.DataFrame:
    """
    Analyze the impact of various improvement measures.
    """
    results = []
    
    for name, wall_type, roof_type in improvements:
        test_flat = FlatProperties(
            floor_area_sqm=base_flat.floor_area_sqm,
            position=base_flat.position,
            wall_type=wall_type,
            roof_type=roof_type,
            floor_type=base_flat.floor_type,
            window_type=base_flat.window_type,
            heating_type=base_flat.heating_type,
            hot_water_type=base_flat.hot_water_type,
            has_room_thermostat=base_flat.has_room_thermostat,
            has_trvs=base_flat.has_trvs,
        )
        
        estimator = EPCEstimator(test_flat)
        result = estimator.calculate_epc()
        heat_loss = estimator.calculate_heat_loss()
        
        results.append({
            'Scenario': name,
            'Wall U-value': estimator.WALL_U_VALUES[wall_type],
            'Roof U-value': estimator.ROOF_U_VALUES[roof_type],
            'Heat Loss (W/K)': round(heat_loss['heat_loss_parameter_wk'], 1),
            'EPC': f"{result['rating_band']} ({result['rating_score']})",
            'Annual Cost (£)': result['estimated_annual_cost_gbp'],
        })
    
    return pd.DataFrame(results)


# Top floor flat with solid walls (worst case scenario)
poor_performance_flat = FlatProperties(
    floor_area_sqm=40,
    position=FlatPosition.TOP_FLOOR,
    wall_type=WallType.SOLID_UNINSULATED,
    roof_type=RoofType.PITCHED_NO_INSULATION,
    floor_type=FloorType.NONE,
    window_type=WindowType.DOUBLE_GLAZED_NEW,
    heating_type=HeatingType.GAS_BOILER_EFFICIENT,
    hot_water_type=HotWaterType.GAS_INSTANT,
    has_room_thermostat=True,
    has_trvs=True,
)

# Define improvement scenarios
improvement_scenarios = [
    ("Baseline (No improvements)", WallType.SOLID_UNINSULATED, RoofType.PITCHED_NO_INSULATION),
    ("Wall: 50mm insulation", WallType.SOLID_50MM_INSULATION, RoofType.PITCHED_NO_INSULATION),
    ("Wall: 100mm insulation", WallType.SOLID_100MM_INSULATION, RoofType.PITCHED_NO_INSULATION),
    ("Roof: 100mm loft insulation", WallType.SOLID_UNINSULATED, RoofType.PITCHED_100MM_LOFT_INSULATION),
    ("Roof: 270mm loft insulation", WallType.SOLID_UNINSULATED, RoofType.PITCHED_270MM_LOFT_INSULATION),
    ("Wall 100mm + Roof 100mm", WallType.SOLID_100MM_INSULATION, RoofType.PITCHED_100MM_LOFT_INSULATION),
    ("Wall 100mm + Roof 270mm", WallType.SOLID_100MM_INSULATION, RoofType.PITCHED_270MM_LOFT_INSULATION),
    ("Wall 150mm + Roof 270mm", WallType.SOLID_150MM_INSULATION, RoofType.PITCHED_270MM_LOFT_INSULATION),
]

In [ ]:
# Run improvement analysis
print("Improvement Analysis for Top Floor Flat with Solid Walls:")
print()
improvement_df = analyze_improvements(poor_performance_flat, improvement_scenarios)
print(improvement_df.to_string(index=False))
print()

# Calculate potential savings
baseline_cost = improvement_df.iloc[0]['Annual Cost (£)']
best_cost = improvement_df.iloc[-1]['Annual Cost (£)']
savings = baseline_cost - best_cost

baseline_epc = improvement_df.iloc[0]['EPC']
best_epc = improvement_df.iloc[-1]['EPC']

print(f"Potential Improvement:")
print(f"  EPC: {baseline_epc} → {best_epc}")
print(f"  Annual Cost: £{baseline_cost} → £{best_cost} (£{savings} savings/year)")

## 7. Visualisation

In [ ]:
def plot_epc_comparison(results, labels):
    """
    Plot comprehensive EPC comparison for multiple flats.
    """
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    
    # EPC Score comparison
    ax1 = axes[0, 0]
    colors = plt.cm.RdYlGn(np.linspace(0.2, 0.8, len(results)))
    bars = ax1.bar(labels, [r['rating_score'] for r in results], color=colors)
    ax1.set_ylabel('EPC Score')
    ax1.set_title('EPC Score Comparison')
    ax1.set_ylim(0, 100)
    ax1.axhline(y=50, color='r', linestyle='--', alpha=0.5, label='D/E Boundary')
    for i, result in enumerate(results):
        ax1.text(i, result['rating_score'] + 2, result['rating_band'], 
                ha='center', va='bottom', fontsize=12, fontweight='bold')
    ax1.legend()
    
    # CO2 Emissions
    ax2 = axes[0, 1]
    ax2.bar(labels, [r['co2_emissions_kg_m2'] for r in results], color=colors)
    ax2.set_ylabel('CO2 Emissions (kg/m²/year)')
    ax2.set_title('CO2 Emissions per m²')
    ax2.tick_params(axis='x', rotation=45)
    
    # Heat Loss Breakdown
    ax3 = axes[0, 2]
    categories = ['Walls', 'Windows', 'Roof', 'Floor']
    x = np.arange(len(labels))
    width = 0.2
    element_colors = ['#e74c3c', '#3498db', '#9b59b6', '#2ecc71']
    
    for i, cat in enumerate(categories):
        key = f"{cat.lower()}_loss_wk"
        values = [r['heat_loss_breakdown'].get(key, 0) for r in results]
        ax3.bar(x + i*width, values, width, label=cat, color=element_colors[i])
    
    ax3.set_ylabel('Heat Loss (W/K)')
    ax3.set_title('Fabric Heat Loss by Element')
    ax3.set_xticks(x + width * 1.5)
    ax3.set_xticklabels(labels, rotation=45)
    ax3.legend()
    
    # Energy breakdown
    ax4 = axes[1, 0]
    categories = ['Space Heating', 'Hot Water', 'Lighting', 'Appliances', 'Cooking']
    x = np.arange(len(labels))
    width = 0.15
    energy_colors = ['#e74c3c', '#e67e22', '#f1c40f', '#3498db', '#2ecc71']
    
    for i, cat in enumerate(categories):
        key_map = ['space_heating_kwh', 'hot_water_kwh', 'lighting_kwh', 'appliance_kwh', 'cooking_kwh']
        values = [r['energy_breakdown'][key_map[i]] for r in results]
        ax4.bar(x + i*width, values, width, label=cat, color=energy_colors[i])
    
    ax4.set_ylabel('Energy (kWh/year)')
    ax4.set_title('Energy Consumption Breakdown')
    ax4.set_xticks(x + width * 2)
    ax4.set_xticklabels(labels, rotation=45)
    ax4.legend(fontsize=8)
    
    # Annual cost
    ax5 = axes[1, 1]
    bars = ax5.bar(labels, [r['estimated_annual_cost_gbp'] for r in results], color=colors)
    ax5.set_ylabel('Annual Cost (£)')
    ax5.set_title('Estimated Annual Energy Cost')
    ax5.tick_params(axis='x', rotation=45)
    for i, result in enumerate(results):
        ax5.text(i, result['estimated_annual_cost_gbp'] + 20, 
                f"£{result['estimated_annual_cost_gbp']}", 
                ha='center', va='bottom', fontsize=9)
    
    # Primary Energy
    ax6 = axes[1, 2]
    ax6.bar(labels, [r['primary_energy_kwh_m2'] for r in results], color=colors)
    ax6.set_ylabel('Primary Energy (kWh/m²/year)')
    ax6.set_title('Primary Energy Demand')
    ax6.tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()


# Prepare labels
labels = [f"Flat {i+1}" for i in range(len(results))]

In [ ]:
# Plot comparison
plot_epc_comparison(results, labels)

## 8. Sensitivity Analysis

In [ ]:
# Analyse impact of different heating systems
base_flat = FlatProperties(
    floor_area_sqm=65,
    position=FlatPosition.MID_FLOOR,
    num_bedrooms=2,
    num_occupants=2,
    year_built=2000,
    wall_type=WallType.CAVITY_FULL_FILL,
    window_type=WindowType.DOUBLE_GLAZED_NEW,
    heating_type=HeatingType.GAS_BOILER_EFFICIENT,
    hot_water_type=HotWaterType.FROM_HEATING,
    has_room_thermostat=True,
    has_trvs=True,
    low_energy_lighting_pct=50,
)

heating_types = [
    HeatingType.GAS_BOILER_OLD,
    HeatingType.GAS_BOILER_EFFICIENT,
    HeatingType.ELECTRIC_DIRECT,
    HeatingType.HEAT_PUMP,
]

print("Heating System Impact on EPC:")
print("-" * 50)
for ht in heating_types:
    test_flat = FlatProperties(
        floor_area_sqm=base_flat.floor_area_sqm,
        position=base_flat.position,
        num_bedrooms=base_flat.num_bedrooms,
        num_occupants=base_flat.num_occupants,
        year_built=base_flat.year_built,
        wall_type=base_flat.wall_type,
        window_type=base_flat.window_type,
        heating_type=ht,
        hot_water_type=base_flat.hot_water_type,
        has_room_thermostat=base_flat.has_room_thermostat,
        has_trvs=base_flat.has_trvs,
        low_energy_lighting_pct=base_flat.low_energy_lighting_pct,
    )
    estimator = EPCEstimator(test_flat)
    result = estimator.calculate_epc()
    print(f"{ht.value:25s}: {result['rating_band']} ({result['rating_score']}) - £{result['estimated_annual_cost_gbp']}/year")

# Create comparison chart
fig, ax = plt.subplots(figsize=(10, 6))
heating_labels = [ht.value.replace('_', ' ').title() for ht in heating_types]
scores = []
for ht in heating_types:
    test_flat = FlatProperties(
        floor_area_sqm=base_flat.floor_area_sqm,
        position=base_flat.position,
        num_bedrooms=base_flat.num_bedrooms,
        num_occupants=base_flat.num_occupants,
        year_built=base_flat.year_built,
        wall_type=base_flat.wall_type,
        window_type=base_flat.window_type,
        heating_type=ht,
        hot_water_type=base_flat.hot_water_type,
        has_room_thermostat=base_flat.has_room_thermostat,
        has_trvs=base_flat.has_trvs,
        low_energy_lighting_pct=base_flat.low_energy_lighting_pct,
    )
    estimator = EPCEstimator(test_flat)
    result = estimator.calculate_epc()
    scores.append(result['rating_score'])

colors = ['#e74c3c', '#f39c12', '#9b59b6', '#27ae60']
bars = ax.bar(heating_labels, scores, color=colors)
ax.set_ylabel('EPC Score')
ax.set_title('EPC Score by Heating System Type')
ax.set_ylim(0, 100)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 9. Interactive Estimator

Modify the values below to estimate your own flat's EPC rating with detailed wall and roof options.

In [ ]:
# Configure your flat properties here
my_flat = FlatProperties(
    floor_area_sqm=60,           # Total floor area in square metres
    position=FlatPosition.MID_FLOOR,
    num_bedrooms=2,
    num_occupants=2,
    year_built=1990,
    
    # Wall configuration
    wall_type=WallType.CAVITY_UNINSULATED,
    num_external_walls=2,
    
    # Roof configuration (for top floor)
    roof_type=RoofType.NONE,
    
    # Floor configuration (for ground floor)
    floor_type=FloorType.NONE,
    
    # Windows
    window_type=WindowType.DOUBLE_GLAZED_OLD,
    
    # Heating
    heating_type=HeatingType.GAS_BOILER_MODERATE,
    hot_water_type=HotWaterType.FROM_HEATING,
    has_cylinder_thermostat=False,
    has_room_thermostat=True,
    has_trvs=True,
    has_programmer=True,
    
    # Lighting and renewables
    low_energy_lighting_pct=30,
    has_solar_pv=False,
    solar_pv_kw=0,
    
    # Party walls/flat separation
    flat_below_insulated=True,
    flat_above_insulated=True,
    party_walls_insulated=False,
    
    # Thermal performance
    thermal_bridge_factor=0.15,
    air_changes_per_hour=0.5,
    has_draught_proofing=False,
)

# Calculate and display results
my_estimator = EPCEstimator(my_flat)
my_result = my_estimator.calculate_epc()

print("=" * 60)
print("YOUR FLAT EPC ESTIMATE")
print("=" * 60)
print(f"\nEPC Rating: {my_result['rating_band']} ({my_result['rating_score']}/100)")
print(f"Primary Energy: {my_result['primary_energy_kwh_m2']} kWh/m²/year")
print(f"CO2 Emissions: {my_result['co2_emissions_kg_m2']} kg/m²/year")
print(f"Total CO2: {my_result['total_co2_kg_year']} kg/year")
print(f"Est. Annual Cost: £{my_result['estimated_annual_cost_gbp']}")

print("\n--- Fabric Summary ---")
fabric_df = my_estimator.get_fabric_summary()
print(fabric_df.to_string(index=False))

print("\n--- Heat Loss Breakdown ---")
hl = my_result['heat_loss_breakdown']
print(f"  Wall Heat Loss: {hl['wall_loss_w']:.0f} W")
print(f"  Window Heat Loss: {hl['window_loss_w']:.0f} W")
print(f"  Roof Heat Loss: {hl['roof_loss_w']:.0f} W")
print(f"  Floor Heat Loss: {hl['floor_loss_w']:.0f} W")
print(f"  Ventilation Loss: {hl['ventilation_loss_w']:.0f} W")
print(f"  Total Heat Loss: {hl['total_loss_w']:.0f} W")

print("\n--- Energy Breakdown ---")
print(f"  Space Heating: {my_result['energy_breakdown']['space_heating_kwh']:,.0f} kWh")
print(f"  Hot Water: {my_result['energy_breakdown']['hot_water_kwh']:,.0f} kWh")
print(f"  Lighting: {my_result['energy_breakdown']['lighting_kwh']:,.0f} kWh")
print(f"  Appliances: {my_result['energy_breakdown']['appliance_kwh']:,.0f} kWh")
print(f"  Cooking: {my_result['energy_breakdown']['cooking_kwh']:,.0f} kWh")
if my_result['energy_breakdown']['solar_generation_kwh'] > 0:
    print(f"  Solar Generation: {my_result['energy_breakdown']['solar_generation_kwh']:,.0f} kWh")
print("=" * 60)

## Notes

This is an enhanced EPC estimator with detailed wall and roof handling. Key improvements include:

### Wall Types
- **Solid walls**: Multiple insulation levels (uninsulated, 50mm, 100mm, 150mm)
- **Cavity walls**: Uninsulated, partial fill, and full fill options
- **Timber frame**: Insulated and uninsulated variants
- **Steel frame**: Insulated and uninsulated variants
- **System built**: Post-war construction types

### Roof Types
- **Pitched roofs**: Various loft insulation levels (0-270mm)
- **Room in roof**: For converted loft spaces
- **Flat roofs**: Standard, warm deck, and inverted configurations

### Floor Types
- **Solid floors**: Insulated and uninsulated
- **Suspended floors**: Timber/concrete with/without insulation

### Additional Features
- Thermal bridging factor (default 15%)
- Adjustable air changes per hour
- Draught proofing option
- Party wall insulation consideration
- Detailed fabric summary output

For an official EPC, visit: https://www.gov.uk/get-new-energy-certificate